In [ ]:
# ---------- Cell 2/3 ----------
# Authorize gspread and define helper functions and parsing logic.

import gspread
from google.auth import default
from datetime import datetime, timedelta, timezone

# Authorize gspread client using application default credentials
creds, _ = default()
gc = gspread.authorize(creds)

# Config - change sheet/worksheet names if needed
GOOGLE_SHEET_NAME = "Buy_Sell_Calls"
GOOGLE_WORKSHEET_NAME = "Sheet1"

# IST helper for timestamps
IST_TZ = timezone(timedelta(hours=5, minutes=30))
def now_ist_str():
    return datetime.now(IST_TZ).strftime("%Y-%m-%d %H:%M:%S")

# Append to Google Sheet
def append_to_sheet(data_row):
    try:
        spreadsheet = gc.open(GOOGLE_SHEET_NAME)
        worksheet = spreadsheet.worksheet(GOOGLE_WORKSHEET_NAME)
        worksheet.append_row(data_row, value_input_option="USER_ENTERED")
        print(f"✅ Added row: {data_row}")
    except Exception as e:
        print(f"❌ Error writing to Google Sheet: {e}")

# Parsing function (expects blank lines between sections)
def parse_stock_message_v2(message_text):
    """
    Expected block format (separated by blank lines):
    1) STOCK NAME
    2) ENTRY_LOW-ENTRY_HIGH  (or single value)
    3) TARGET1 (newline) TARGET2 (newline) TARGET3  (1-3 lines)
    4) STOPLOSS
    5) SOURCE
    6) TYPE
    """
    blocks = [b.strip() for b in message_text.strip().split("\n\n") if b.strip()]
    if len(blocks) < 6:
        return None, "❌ Missing one or more sections. Please follow the exact format."

    try:
        stock_name = blocks[0].title()

        # Entry range (allow single value or low-high)
        entry_block = blocks[1].replace(" ", "")
        if "-" in entry_block:
            entry_low_s, entry_high_s = entry_block.split("-", 1)
        else:
            entry_low_s = entry_high_s = entry_block
        entry_low, entry_high = float(entry_low_s), float(entry_high_s)

        # Targets: allow 1-3 lines, pad with blanks
        target_lines = [t.strip() for t in blocks[2].splitlines() if t.strip()]
        if not 1 <= len(target_lines) <= 3:
            return None, "❌ You must enter between 1 and 3 targets (each on a new line)."
        targets = [float(t) for t in target_lines]
        while len(targets) < 3:
            targets.append("")

        stoploss = float(blocks[3])
        source = blocks[4].title()
        trade_type = blocks[5].title()

        parsed = [
            stock_name,
            entry_low,
            entry_high,
            targets[0],
            targets[1],
            targets[2],
            stoploss,
            source,
            trade_type,
        ]
        return parsed, None
    except Exception as e:
        return None, f"❌ Parsing error: {e}"

print("✔ Helpers and parser ready.")
